In [ ]:
%run 01_inventory_airbnb.py

In [ ]:
import pandas as pd

inventory = pd.read_csv(INVENTORY / "airbnb_inventory.csv") if "INVENTORY" in globals() else pd.read_csv(INTERMEDIATE / "airbnb_inventory.csv")

display(
    inventory[
        ["snapshot_date", "file_name", "size_mb", "row_count", "column_count"]
    ]
)

In [ ]:
import pandas as pd

cols = pd.read_csv(
    INTERMEDIATE / "airbnb_columns_by_snapshot.csv"
)

listing_cols = cols[
    cols["file_name"] == "listings.csv.gz"
].copy()

column_sets = {
    date: set(
        listing_cols.loc[
            listing_cols["snapshot_date"] == date,
            "column_name"
        ]
    )
    for date in SNAPSHOT_DATES
}

base = column_sets["2025-10-05"]

print("=== LISTINGS SCHEMA CHANGES ===")

for date in SNAPSHOT_DATES:
    current = column_sets[date]

    added = sorted(current - base)
    removed = sorted(base - current)

    print(f"\n{date}")
    print(f"Total columns: {len(current)}")
    print("Added vs 2025-10-05:", added)
    print("Removed vs 2025-10-05:", removed)

In [ ]:
import pandas as pd

cols = pd.read_csv(
    INTERMEDIATE / "airbnb_columns_by_snapshot.csv"
)

core_85 = (
    cols[
        (cols["snapshot_date"] == "2025-10-05") &
        (cols["file_name"] == "listings.csv.gz")
    ]
    [["column_position", "column_name"]]
    .sort_values("column_position")
    .reset_index(drop=True)
)

print("Number of common-core columns:", len(core_85))
display(core_85)

In [ ]:
display(core_85)

In [ ]:
pd.set_option("display.max_rows", None)

display(core_85)

In [ ]:
for _, row in core_85.iterrows():
    print(f"{int(row['column_position']):02d}. {row['column_name']}")

In [ ]:
import pandas as pd

# ============================================================
# CANDIDATE LISTINGS VARIABLES FOR INITIAL QC
# ============================================================

candidate_cols = [
    # ID / time
    "id",
    "last_scraped",

    # Geography
    "latitude",
    "longitude",

    # Property characteristics
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bathrooms_text",
    "bedrooms",
    "beds",
    "amenities",

    # Host / platform characteristics
    "host_since",
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate",
    "host_is_superhost",
    "host_listings_count",
    "host_total_listings_count",
    "host_has_profile_pic",
    "host_identity_verified",
    "instant_bookable",
    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",

    # Listing conditions / advertised price
    "price",
    "minimum_nights",
    "maximum_nights",
    "has_availability",

    # Historical review information
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "first_review",
    "last_review",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "reviews_per_month",
]

test_file = RAW_AIRBNB / "2025-10-05" / "listings.csv.gz"

df_test = pd.read_csv(
    test_file,
    usecols=candidate_cols,
    low_memory=False
)

print("Snapshot: 2025-10-05")
print("Rows:", len(df_test))
print("Candidate columns:", len(df_test.columns))

qc = pd.DataFrame({
    "column": df_test.columns,
    "dtype": df_test.dtypes.astype(str).values,
    "missing_n": df_test.isna().sum().values,
    "missing_pct": (
        df_test.isna().mean().values * 100
    ).round(2)
})

display(qc)

In [ ]:
import pandas as pd

test_file = RAW_AIRBNB / "2025-10-05" / "listings.csv.gz"

df_count_check = pd.read_csv(
    test_file,
    usecols=["id"],
    low_memory=False
)

print("Total CSV records :", len(df_count_check))
print("Unique listing IDs:", df_count_check["id"].nunique())
print("Duplicate IDs     :", df_count_check["id"].duplicated().sum())
print("Missing IDs       :", df_count_check["id"].isna().sum())

In [ ]:
%run 01_inventory_airbnb.py

In [ ]:
candidate_cols


In [ ]:
# ============================================================
# FULL 85-VARIABLE SCREENING TABLE
# ============================================================

screening_85 = core_85.copy()

screening_85["decision"] = "TO_REVIEW"
screening_85["role"] = ""
screening_85["reason"] = ""
screening_85["planned_transformation"] = ""
screening_85["leakage_concern"] = ""

print("Total variables:", len(screening_85))

display(screening_85)

In [ ]:
# ============================================================
# STEP 10.2B — SYSTEMATIC SCREENING OF 85 COMMON VARIABLES
# ============================================================

KEEP = {
    # Property characteristics
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",

    # Host / platform characteristics
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate",
    "host_is_superhost",
    "host_listings_count",
    "host_total_listings_count",
    "host_has_profile_pic",
    "host_identity_verified",
    "instant_bookable",
    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",

    # Listing conditions
    "minimum_nights",
    "maximum_nights",

    # Historical review state
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "reviews_per_month",
}

DERIVE = {
    "host_since",       # -> host tenure at anchor
    "amenities",        # -> amenities_count
    "first_review",     # -> review-history features if retained
    "last_review",      # -> recency features
}

PROCESSING_ONLY = {
    "id",               # listing identifier / temporal matching
    "last_scraped",     # temporal QC
    "latitude",         # TA spatial assignment
    "longitude",        # TA spatial assignment
    "price",            # advertised-price construction / outcome
}

EXCLUDE = set(screening_85["column_name"]) - (
    KEEP | DERIVE | PROCESSING_ONLY
)


def screening_details(variable):

    if variable in KEEP:
        return (
            "KEEP",
            "Candidate predictor",
            "Scientifically relevant property, host/platform, listing-condition, "
            "or historical review-state information available in the common temporal schema.",
            "Clean/standardise as appropriate",
            "Use anchor-time information only"
        )

    if variable == "host_since":
        return (
            "DERIVE",
            "Host history",
            "Raw date is more meaningfully represented as host tenure at each anchor.",
            "Derive host_tenure_days/months using anchor date",
            "Do not calculate using future snapshot date"
        )

    if variable == "amenities":
        return (
            "DERIVE",
            "Property characteristics",
            "Raw amenities text/list is high dimensional; a compact reproducible summary is preferred.",
            "Derive amenities_count",
            "Anchor snapshot only"
        )

    if variable == "first_review":
        return (
            "DERIVE",
            "Historical review state",
            "Raw date may provide listing review-history information rather than serving as a direct predictor.",
            "Derive historical review-age feature if retained",
            "Anchor-time information only"
        )

    if variable == "last_review":
        return (
            "DERIVE",
            "Historical review state",
            "Review recency is more interpretable than the raw date.",
            "Derive days_since_last_review",
            "Use reviews available by anchor only"
        )

    if variable == "id":
        return (
            "PROCESSING_ONLY",
            "Identifier",
            "Required to match the same listing across temporal snapshots and construct future outcomes.",
            "Rename to listing_id",
            "Never use identifier itself as predictor"
        )

    if variable == "last_scraped":
        return (
            "PROCESSING_ONLY",
            "Temporal metadata",
            "Useful for QC of snapshot timing but not substantive predictive information.",
            "Parse as date for QC",
            "Do not use as predictor"
        )

    if variable in {"latitude", "longitude"}:
        return (
            "PROCESSING_ONLY",
            "Geography",
            "Required for point-in-polygon assignment to the fixed 2025 Territorial Authority geography.",
            "Spatial join to TA 2025",
            "Use derived TA/location representation in modelling"
        )

    if variable == "price":
        return (
            "PROCESSING_ONLY",
            "Advertised price",
            "Required for advertised-price processing and price-resilience outcome construction; "
            "it is not interpreted as realized revenue.",
            "Parse currency to numeric",
            "Respect anchor/future target separation"
        )

    # Specific exclusion reasons
    if variable in {
        "listing_url", "picture_url", "host_url",
        "host_thumbnail_url", "host_picture_url"
    }:
        return (
            "EXCLUDE",
            "URL / identifier metadata",
            "URL metadata has no required role in the predictive framework.",
            "None",
            "Not used"
        )

    if variable in {
        "host_name", "host_about", "name", "description",
        "neighborhood_overview"
    }:
        return (
            "EXCLUDE",
            "Free text / personal metadata",
            "Not required for the predefined structured property/platform branch; "
            "guest-experience semantics are derived separately from review text.",
            "None",
            "Not used"
        )

    if variable in {
        "estimated_occupancy_l365d",
        "estimated_revenue_l365d"
    }:
        return (
            "EXCLUDE",
            "Provider-derived estimate",
            "Excluded to avoid treating provider-estimated occupancy or revenue as observed ground truth.",
            "None",
            "Do not interpret as realized occupancy/revenue"
        )

    if variable.startswith("availability_") or variable == "has_availability":
        return (
            "EXCLUDE",
            "Availability summary",
            "Main calendar-configuration features will be constructed consistently from detailed calendar data.",
            "None",
            "Do not interpret availability as bookings or occupancy"
        )

    if variable.startswith("region_") or variable in {
        "neighbourhood", "host_neighbourhood", "host_location"
    }:
        return (
            "EXCLUDE",
            "Provider geography",
            "Final geography will be assigned consistently using latitude/longitude and fixed TA 2025 boundaries.",
            "None",
            "Avoid inconsistent provider geography"
        )

    return (
        "EXCLUDE",
        "Non-essential raw metadata",
        "Not required for the predefined modelling framework or reproducible feature construction.",
        "None",
        "Not used"
    )


screening_85[
    [
        "decision",
        "role",
        "reason",
        "planned_transformation",
        "leakage_concern"
    ]
] = screening_85["column_name"].apply(
    lambda x: pd.Series(screening_details(x))
)


# QC
print("=== SCREENING SUMMARY ===")
print(screening_85["decision"].value_counts())
print("\nTotal:", len(screening_85))

display(screening_85)

In [ ]:
# ============================================================
# STEP 10.2C — SAVE INITIAL 85-VARIABLE SCREENING
# ============================================================

screening_output = INTERMEDIATE / "airbnb_listings_85_variable_screening.csv"

screening_85.to_csv(
    screening_output,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(screening_output)

print("\nQC:")
print("Rows:", len(screening_85))
print("Unique variables:", screening_85["column_name"].nunique())
print("Remaining TO_REVIEW:", (screening_85["decision"] == "TO_REVIEW").sum())

In [ ]:
# ============================================================
# STEP 10.3A — TEMPORAL DATA-QUALITY AUDIT
# 41 non-excluded variables across all 11 snapshots
# ============================================================

audit_vars = screening_85.loc[
    screening_85["decision"].isin(
        ["KEEP", "DERIVE", "PROCESSING_ONLY"]
    ),
    "column_name"
].tolist()

print("Variables to audit:", len(audit_vars))

audit_results = []

for snapshot_date in SNAPSHOT_DATES:

    print(f"Reading {snapshot_date} ...")

    file_path = (
        RAW_AIRBNB /
        snapshot_date /
        "listings.csv.gz"
    )

    df = pd.read_csv(
        file_path,
        usecols=audit_vars,
        low_memory=False
    )

    n_rows = len(df)

    for col in audit_vars:

        missing_n = int(df[col].isna().sum())
        missing_pct = round(
            100 * missing_n / n_rows,
            2
        )

        audit_results.append({
            "snapshot_date": snapshot_date,
            "n_listings": n_rows,
            "column_name": col,
            "dtype": str(df[col].dtype),
            "missing_n": missing_n,
            "missing_pct": missing_pct
        })

    del df


temporal_audit = pd.DataFrame(audit_results)


print("\n=== QC ===")
print("Snapshots:", temporal_audit["snapshot_date"].nunique())
print("Variables:", temporal_audit["column_name"].nunique())
print("Audit rows:", len(temporal_audit))

In [ ]:
# ============================================================
# STEP 10.3B — TEMPORAL MISSINGNESS SUMMARY
# ============================================================

missing_summary = (
    temporal_audit
    .groupby("column_name")
    .agg(
        min_missing_pct=("missing_pct", "min"),
        mean_missing_pct=("missing_pct", "mean"),
        max_missing_pct=("missing_pct", "max")
    )
    .reset_index()
)

missing_summary["mean_missing_pct"] = (
    missing_summary["mean_missing_pct"].round(2)
)

# Sort from highest maximum missingness to lowest
missing_summary = missing_summary.sort_values(
    ["max_missing_pct", "mean_missing_pct"],
    ascending=False
).reset_index(drop=True)

print("=== TEMPORAL MISSINGNESS SUMMARY ===")
print("Variables:", len(missing_summary))

display(missing_summary)

In [ ]:
# ============================================================
# STEP 10.3C — INVESTIGATE TEMPORAL MISSINGNESS ANOMALIES
# ============================================================

problem_vars = [
    "host_response_rate",
    "host_response_time",
    "host_acceptance_rate",
    "host_since",
    "host_total_listings_count",
    "instant_bookable",
    "price"
]

problem_missingness = (
    temporal_audit[
        temporal_audit["column_name"].isin(problem_vars)
    ]
    .pivot(
        index="column_name",
        columns="snapshot_date",
        values="missing_pct"
    )
)

problem_missingness = problem_missingness[
    SNAPSHOT_DATES
]

display(problem_missingness)

In [ ]:
# ============================================================
# STEP 10.3D — CHECK CALENDAR PRICE AVAILABILITY
# Problem snapshot: 2025-12-11
# ============================================================

calendar_test_file = (
    RAW_AIRBNB /
    "2025-12-11" /
    "calendar.csv.gz"
)

calendar_sample = pd.read_csv(
    calendar_test_file,
    nrows=10
)

print("Calendar columns:")
print(calendar_sample.columns.tolist())

print("\nSample:")
display(calendar_sample)

In [ ]:
# ============================================================
# STEP 10.3E — FULL CALENDAR PRICE AVAILABILITY CHECK
# Snapshot: 2025-12-11
# Memory-safe chunk processing
# ============================================================

calendar_test_file = (
    RAW_AIRBNB /
    "2025-12-11" /
    "calendar.csv.gz"
)

total_rows = 0
price_nonmissing = 0
adjusted_price_nonmissing = 0

for chunk in pd.read_csv(
    calendar_test_file,
    usecols=["price", "adjusted_price"],
    chunksize=500_000,
    low_memory=False
):
    total_rows += len(chunk)
    price_nonmissing += chunk["price"].notna().sum()
    adjusted_price_nonmissing += chunk["adjusted_price"].notna().sum()

print("=== CALENDAR PRICE QC: 2025-12-11 ===")
print(f"Total rows: {total_rows:,}")

print(
    f"price non-missing: "
    f"{price_nonmissing:,} "
    f"({100 * price_nonmissing / total_rows:.2f}%)"
)

print(
    f"adjusted_price non-missing: "
    f"{adjusted_price_nonmissing:,} "
    f"({100 * adjusted_price_nonmissing / total_rows:.2f}%)"
)

In [ ]:
# ============================================================
# STEP 10.3F — CHECK PRICE AVAILABILITY
# 2026-01-16 and 2026-02-13
# ============================================================

check_dates = [
    "2026-01-16",
    "2026-02-13"
]

results = []

for snapshot_date in check_dates:

    calendar_file = (
        RAW_AIRBNB /
        snapshot_date /
        "calendar.csv.gz"
    )

    total_rows = 0
    price_nonmissing = 0
    adjusted_price_nonmissing = 0

    print(f"Checking {snapshot_date} ...")

    for chunk in pd.read_csv(
        calendar_file,
        usecols=["price", "adjusted_price"],
        chunksize=500_000,
        low_memory=False
    ):
        total_rows += len(chunk)

        price_nonmissing += int(
            chunk["price"].notna().sum()
        )

        adjusted_price_nonmissing += int(
            chunk["adjusted_price"].notna().sum()
        )

    results.append({
        "snapshot_date": snapshot_date,
        "total_rows": total_rows,
        "price_nonmissing": price_nonmissing,
        "price_nonmissing_pct":
            round(100 * price_nonmissing / total_rows, 2),
        "adjusted_price_nonmissing":
            adjusted_price_nonmissing,
        "adjusted_price_nonmissing_pct":
            round(
                100 * adjusted_price_nonmissing / total_rows,
                2
            )
    })


price_gap_check = pd.DataFrame(results)

display(price_gap_check)

In [ ]:
# ============================================================
# STEP 10.3G — SCREEN CALENDAR PRICE AVAILABILITY
# ALL 11 SNAPSHOTS
# ============================================================

calendar_price_screen = []

for snapshot_date in SNAPSHOT_DATES:

    calendar_file = (
        RAW_AIRBNB /
        snapshot_date /
        "calendar.csv.gz"
    )

    # Read first 200,000 records only for rapid screening
    sample = pd.read_csv(
        calendar_file,
        usecols=["price", "adjusted_price"],
        nrows=200_000,
        low_memory=False
    )

    price_n = int(sample["price"].notna().sum())
    adjusted_n = int(sample["adjusted_price"].notna().sum())

    calendar_price_screen.append({
        "snapshot_date": snapshot_date,
        "sample_rows": len(sample),
        "price_nonmissing_pct":
            round(100 * price_n / len(sample), 2),
        "adjusted_price_nonmissing_pct":
            round(100 * adjusted_n / len(sample), 2)
    })

    print(
        snapshot_date,
        "| price:",
        round(100 * price_n / len(sample), 2),
        "%"
    )

calendar_price_screen = pd.DataFrame(
    calendar_price_screen
)

display(calendar_price_screen)

In [ ]:
# ============================================================
# STEP 10.3H — FULL CALENDAR PRICE CHECK
# Control snapshot: 2026-08-13
# ============================================================

snapshot_date = "2026-08-13"

calendar_file = (
    RAW_AIRBNB /
    snapshot_date /
    "calendar.csv.gz"
)

total_rows = 0
price_nonmissing = 0
adjusted_price_nonmissing = 0

for chunk in pd.read_csv(
    calendar_file,
    usecols=["price", "adjusted_price"],
    chunksize=500_000,
    low_memory=False
):
    total_rows += len(chunk)
    price_nonmissing += int(
        chunk["price"].notna().sum()
    )
    adjusted_price_nonmissing += int(
        chunk["adjusted_price"].notna().sum()
    )

print("=== FULL CALENDAR PRICE QC ===")
print("Snapshot:", snapshot_date)
print(f"Total rows: {total_rows:,}")

print(
    f"price non-missing: {price_nonmissing:,} "
    f"({100 * price_nonmissing / total_rows:.2f}%)"
)

print(
    f"adjusted_price non-missing: "
    f"{adjusted_price_nonmissing:,} "
    f"({100 * adjusted_price_nonmissing / total_rows:.2f}%)"
)

In [ ]:
# ============================================================
# STEP 10.3I — INSPECT LISTINGS.PRICE BEFORE/AFTER DATA GAP
# ============================================================

check_dates = [
    "2025-10-05",
    "2026-08-13"
]

for snapshot_date in check_dates:

    file_path = (
        RAW_AIRBNB /
        snapshot_date /
        "listings.csv.gz"
    )

    df_price = pd.read_csv(
        file_path,
        usecols=["id", "price"],
        low_memory=False
    )

    print("\n" + "=" * 60)
    print("SNAPSHOT:", snapshot_date)
    print("=" * 60)

    print("Rows:", len(df_price))
    print("Non-missing price:", df_price["price"].notna().sum())
    print(
        "Missing %:",
        round(df_price["price"].isna().mean() * 100, 2)
    )

    print("\nFirst 20 non-missing raw price values:")
    print(
        df_price.loc[
            df_price["price"].notna(),
            "price"
        ].head(20).tolist()
    )

In [ ]:
# ============================================================
# STEP 10.3J — OFFICIAL LISTINGS.PRICE AVAILABILITY TABLE
# ALL 11 SNAPSHOTS
# ============================================================

price_availability = []

for snapshot_date in SNAPSHOT_DATES:

    file_path = (
        RAW_AIRBNB /
        snapshot_date /
        "listings.csv.gz"
    )

    df_price = pd.read_csv(
        file_path,
        usecols=["id", "price"],
        low_memory=False
    )

    total = len(df_price)
    nonmissing = int(df_price["price"].notna().sum())
    missing = int(df_price["price"].isna().sum())

    price_availability.append({
        "snapshot_date": snapshot_date,
        "n_listings": total,
        "price_nonmissing_n": nonmissing,
        "price_nonmissing_pct":
            round(100 * nonmissing / total, 2),
        "price_missing_n": missing,
        "price_missing_pct":
            round(100 * missing / total, 2)
    })

    del df_price


price_availability = pd.DataFrame(price_availability)

display(price_availability)

In [ ]:
# ============================================================
# STEP 10.3K — SAVE TEMPORAL DATA-QUALITY AUDIT
# ============================================================

temporal_audit_file = (
    INTERMEDIATE /
    "airbnb_listings_temporal_missingness_audit.csv"
)

price_availability_file = (
    INTERMEDIATE /
    "airbnb_listings_price_availability.csv"
)

temporal_audit.to_csv(
    temporal_audit_file,
    index=False,
    encoding="utf-8-sig"
)

price_availability.to_csv(
    price_availability_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(temporal_audit_file)
print(price_availability_file)

print("\nQC:")
print("Temporal audit rows:", len(temporal_audit))
print("Price availability rows:", len(price_availability))
print(
    "Snapshots with 100% missing price:",
    (price_availability["price_missing_pct"] == 100).sum()
)

In [ ]:
# ============================================================
# STEP 10.4A — UPDATE SCREENING AFTER TEMPORAL AUDIT
# Structural availability break from Jan 2026
# ============================================================

structural_break_vars = [
    "host_acceptance_rate",
    "host_response_rate",
    "host_response_time",
    "host_since",
    "host_total_listings_count",
    "instant_bookable"
]

mask = screening_85["column_name"].isin(
    structural_break_vars
)

screening_85.loc[
    mask, "decision"
] = "EXCLUDE_TEMPORAL_INCONSISTENCY"

screening_85.loc[
    mask, "role"
] = "Temporally inconsistent field"

screening_85.loc[
    mask, "reason"
] = (
    "Field is available in early snapshots but becomes "
    "100% missing from 2026-01-16 onward; excluded from "
    "the longitudinal modelling feature set to preserve "
    "temporal comparability."
)

screening_85.loc[
    mask, "planned_transformation"
] = "None"

screening_85.loc[
    mask, "leakage_concern"
] = (
    "Do not impute across the structural availability break; "
    "missingness could encode snapshot/data-provider regime."
)


print("=== UPDATED STRUCTURAL-BREAK VARIABLES ===")

display(
    screening_85.loc[
        mask,
        [
            "column_name",
            "decision",
            "reason",
            "leakage_concern"
        ]
    ]
)

print("\nNumber updated:", mask.sum())

In [ ]:
# ============================================================
# STEP 10.4B — PRICE-OUTCOME FEASIBILITY BY TRANSITION
# ============================================================

price_pct = dict(
    zip(
        price_availability["snapshot_date"],
        price_availability["price_nonmissing_pct"]
    )
)

transition_results = []

for i in range(len(SNAPSHOT_DATES) - 1):

    anchor = SNAPSHOT_DATES[i]
    future = SNAPSHOT_DATES[i + 1]

    anchor_available = price_pct[anchor] > 0
    future_available = price_pct[future] > 0

    feasible = anchor_available and future_available

    transition_results.append({
        "transition": f"T{i+1}",
        "anchor_date": anchor,
        "future_date": future,
        "anchor_price_available_pct": price_pct[anchor],
        "future_price_available_pct": price_pct[future],
        "price_resilience_feasible": feasible
    })

price_transition_qc = pd.DataFrame(transition_results)

display(price_transition_qc)

print("\n=== SUMMARY ===")
print(
    "Feasible transitions:",
    price_transition_qc["price_resilience_feasible"].sum()
)
print(
    "Non-feasible transitions:",
    (~price_transition_qc["price_resilience_feasible"]).sum()
)

In [ ]:
# ============================================================
# STEP 10.4C — DOCUMENT PRICE-OUTCOME ELIGIBILITY
# ============================================================

price_transition_qc["analysis_status"] = (
    price_transition_qc["price_resilience_feasible"]
    .map({
        True: "ELIGIBLE_SECONDARY_PRICE_ANALYSIS",
        False: "EXCLUDED_PRICE_DATA_UNAVAILABLE"
    })
)

price_transition_qc["exclusion_reason"] = ""

mask_unavailable = (
    ~price_transition_qc["price_resilience_feasible"]
)

price_transition_qc.loc[
    mask_unavailable,
    "exclusion_reason"
] = (
    "Advertised price is completely unavailable in at least "
    "one of the two snapshots; no temporal interpolation or "
    "synthetic price target is used."
)

display(
    price_transition_qc[
        [
            "transition",
            "anchor_date",
            "future_date",
            "price_resilience_feasible",
            "analysis_status",
            "exclusion_reason"
        ]
    ]
)

print("\n=== QC ===")
print(
    price_transition_qc["analysis_status"].value_counts()
)

In [ ]:
# ============================================================
# STEP 10.4D — SAVE TEMPORAL SCREENING DECISIONS
# ============================================================

# 1. Save price-transition eligibility
price_transition_file = (
    INTERMEDIATE /
    "airbnb_price_transition_eligibility.csv"
)

price_transition_qc.to_csv(
    price_transition_file,
    index=False,
    encoding="utf-8-sig"
)


# 2. Save updated 85-variable screening table
screening_file = (
    INTERMEDIATE /
    "airbnb_listings_85_variable_screening.csv"
)

screening_85.to_csv(
    screening_file,
    index=False,
    encoding="utf-8-sig"
)


# 3. QC
print("Saved:")
print(price_transition_file)
print(screening_file)

print("\n=== QC ===")
print("Screening rows:", len(screening_85))
print(
    "Unique variables:",
    screening_85["column_name"].nunique()
)

print(
    "Temporal inconsistency exclusions:",
    (
        screening_85["decision"]
        == "EXCLUDE_TEMPORAL_INCONSISTENCY"
    ).sum()
)

print(
    "Eligible price transitions:",
    price_transition_qc[
        "price_resilience_feasible"
    ].sum()
)

print(
    "Excluded price transitions:",
    (
        ~price_transition_qc[
            "price_resilience_feasible"
        ]
    ).sum()
)

In [ ]:
# ============================================================
# STEP 10.5A — REVIEW FINAL SCREENING DECISIONS
# ============================================================

decision_summary = (
    screening_85["decision"]
    .value_counts(dropna=False)
    .rename_axis("decision")
    .reset_index(name="n_variables")
)

print("=== DECISION SUMMARY ===")
display(decision_summary)

print("\nTotal variables:", decision_summary["n_variables"].sum())

In [ ]:
# ============================================================
# STEP 10.5B — REVIEW KEEP + DERIVE VARIABLES
# ============================================================

model_candidate_review = (
    screening_85[
        screening_85["decision"].isin(
            ["KEEP", "DERIVE"]
        )
    ][
        [
            "column_position",
            "column_name",
            "decision",
            "role",
            "reason",
            "planned_transformation",
            "leakage_concern"
        ]
    ]
    .sort_values("column_position")
    .reset_index(drop=True)
)

print("=== KEEP + DERIVE VARIABLES ===")
print("Total:", len(model_candidate_review))
print(
    "KEEP:",
    (model_candidate_review["decision"] == "KEEP").sum()
)
print(
    "DERIVE:",
    (model_candidate_review["decision"] == "DERIVE").sum()
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)

display(model_candidate_review)

In [ ]:
# ============================================================
# STEP 10.5C — REVIEW PROCESSING-ONLY VARIABLES
# ============================================================

processing_review = (
    screening_85[
        screening_85["decision"] == "PROCESSING_ONLY"
    ][
        [
            "column_position",
            "column_name",
            "decision",
            "role",
            "reason",
            "planned_transformation",
            "leakage_concern"
        ]
    ]
    .sort_values("column_position")
    .reset_index(drop=True)
)

print("=== PROCESSING-ONLY VARIABLES ===")
print("Total:", len(processing_review))

display(processing_review)

In [ ]:
# ============================================================
# STEP 10.5D — REVIEW ALL EXCLUDED VARIABLES
# ============================================================

excluded_review = (
    screening_85[
        screening_85["decision"] == "EXCLUDE"
    ][
        [
            "column_position",
            "column_name",
            "role",
            "reason",
            "leakage_concern"
        ]
    ]
    .sort_values("column_position")
    .reset_index(drop=True)
)

print("=== EXCLUDED VARIABLES REVIEW ===")
print("Total:", len(excluded_review))

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)

display(excluded_review)

In [ ]:
# ============================================================
# STEP 10.5E — CHECK REVIEW-COUNT REDUNDANCY
# number_of_reviews_ly vs number_of_reviews_ltm
# ============================================================

check_dates = [
    "2025-10-05",
    "2026-03-17",
    "2026-08-13"
]

for snapshot_date in check_dates:

    file_path = (
        RAW_AIRBNB /
        snapshot_date /
        "listings.csv.gz"
    )

    df_check = pd.read_csv(
        file_path,
        usecols=[
            "number_of_reviews_ly",
            "number_of_reviews_ltm"
        ],
        low_memory=False
    )

    both = df_check[
        ["number_of_reviews_ly", "number_of_reviews_ltm"]
    ].dropna()

    same_pct = (
        (both["number_of_reviews_ly"]
         == both["number_of_reviews_ltm"])
        .mean() * 100
    )

    correlation = (
        both["number_of_reviews_ly"]
        .corr(both["number_of_reviews_ltm"])
    )

    print("\n" + "=" * 55)
    print("Snapshot:", snapshot_date)
    print("Rows compared:", len(both))
    print(f"Exactly identical: {same_pct:.2f}%")
    print(f"Correlation: {correlation:.4f}")

In [ ]:
# ============================================================
# STEP 10.5F — INSPECT REVIEW COUNT VARIABLES
# ============================================================

snapshot_date = "2025-10-05"

file_path = (
    RAW_AIRBNB /
    snapshot_date /
    "listings.csv.gz"
)

review_count_check = pd.read_csv(
    file_path,
    usecols=[
        "id",
        "number_of_reviews",
        "number_of_reviews_ltm",
        "number_of_reviews_ly",
        "number_of_reviews_l30d"
    ],
    low_memory=False
)

print("Snapshot:", snapshot_date)
print("Rows:", len(review_count_check))

display(
    review_count_check.head(30)
)

print("\n=== SUMMARY ===")

display(
    review_count_check[
        [
            "number_of_reviews",
            "number_of_reviews_ltm",
            "number_of_reviews_ly",
            "number_of_reviews_l30d"
        ]
    ].describe().round(2)
)

In [ ]:
# ============================================================
# STEP 10.5G — FINALISE number_of_reviews_ly DECISION
# ============================================================

mask_ly = (
    screening_85["column_name"]
    == "number_of_reviews_ly"
)

screening_85.loc[
    mask_ly, "decision"
] = "EXCLUDE"

screening_85.loc[
    mask_ly, "role"
] = "Overlapping historical review activity"

screening_85.loc[
    mask_ly, "reason"
] = (
    "Excluded from the predefined structured feature set. "
    "The field is not identical to number_of_reviews_ltm, "
    "but shows substantial overlap with retained review-activity "
    "measures (number_of_reviews, number_of_reviews_ltm, and "
    "number_of_reviews_l30d), while its distinct temporal "
    "interpretation is not required for the modelling framework."
)

screening_85.loc[
    mask_ly, "planned_transformation"
] = "None"

screening_85.loc[
    mask_ly, "leakage_concern"
] = (
    "Use only clearly defined anchor-time review-history measures."
)

display(
    screening_85.loc[
        mask_ly,
        [
            "column_name",
            "decision",
            "role",
            "reason",
            "leakage_concern"
        ]
    ]
)

print("\nRows updated:", mask_ly.sum())

In [ ]:
# ============================================================
# STEP 10.5H — FREEZE FINAL LISTINGS INPUT SCHEMA
# ============================================================

final_input_schema = (
    screening_85[
        screening_85["decision"].isin(
            ["KEEP", "DERIVE", "PROCESSING_ONLY"]
        )
    ][
        [
            "column_position",
            "column_name",
            "decision",
            "role",
            "planned_transformation",
            "leakage_concern"
        ]
    ]
    .sort_values("column_position")
    .reset_index(drop=True)
)

print("=== FINAL LISTINGS INPUT SCHEMA ===")

print("Total raw columns to read:", len(final_input_schema))

print(
    "\nDecision counts:"
)

print(
    final_input_schema["decision"].value_counts()
)

print("\nExcluded from raw cleaning input:")

print(
    len(screening_85) -
    len(final_input_schema)
)

display(final_input_schema)

In [ ]:
# ============================================================
# STEP 10.5I — SAVE AND FREEZE FINAL LISTINGS SCHEMA
# ============================================================

# ------------------------------------------------------------
# 1. Save the complete updated 85-variable screening table
# ------------------------------------------------------------

screening_file = (
    INTERMEDIATE /
    "airbnb_listings_85_variable_screening.csv"
)

screening_85.to_csv(
    screening_file,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 2. Save final 35-column raw input schema
# ------------------------------------------------------------

schema_file = (
    INTERMEDIATE /
    "airbnb_listings_final_input_schema.csv"
)

final_input_schema.to_csv(
    schema_file,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 3. QC
# ------------------------------------------------------------

print("=== FINAL SCHEMA FREEZE QC ===")

print("Screening variables:", len(screening_85))
print(
    "Unique screening variables:",
    screening_85["column_name"].nunique()
)

print(
    "Final raw input columns:",
    len(final_input_schema)
)

print(
    "KEEP:",
    (final_input_schema["decision"] == "KEEP").sum()
)

print(
    "DERIVE:",
    (final_input_schema["decision"] == "DERIVE").sum()
)

print(
    "PROCESSING_ONLY:",
    (
        final_input_schema["decision"]
        == "PROCESSING_ONLY"
    ).sum()
)

print(
    "Temporal inconsistency exclusions:",
    (
        screening_85["decision"]
        == "EXCLUDE_TEMPORAL_INCONSISTENCY"
    ).sum()
)

print(
    "Standard exclusions:",
    (
        screening_85["decision"]
        == "EXCLUDE"
    ).sum()
)

print("\nSaved:")
print(screening_file)
print(schema_file)

In [ ]:
%run 02_clean_listings.py

In [ ]:
# ============================================================
# STEP 11.3 — INVESTIGATE REVIEW-DATE TEMPORAL MISMATCH
# ============================================================

test_parquet = (
    INT_LISTINGS /
    "listings_2025-10-05.parquet"
)

qc = pd.read_parquet(
    test_parquet,
    columns=[
        "listing_id",
        "snapshot_date",
        "last_scraped",
        "first_review",
        "last_review",
        "listing_review_age_days",
        "days_since_last_review"
    ]
)

print("=== SNAPSHOT DATE ===")
print(qc["snapshot_date"].value_counts(dropna=False))


print("\n=== LAST_SCRAPED DISTRIBUTION ===")

display(
    qc["last_scraped"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("n_listings")
)


print("\n=== REVIEW DATES AFTER SNAPSHOT DATE ===")

problem = qc[
    (qc["listing_review_age_days"] < 0) |
    (qc["days_since_last_review"] < 0)
].copy()

print("Problem rows:", len(problem))

display(
    problem.sort_values(
        "last_review",
        ascending=False
    ).head(30)
)


print("\n=== KEY QC ===")

print(
    "last_scraped after nominal snapshot date:",
    (qc["last_scraped"] > qc["snapshot_date"]).sum()
)

print(
    "last_review after nominal snapshot date:",
    (qc["last_review"] > qc["snapshot_date"]).sum()
)

print(
    "last_review after actual last_scraped:",
    (qc["last_review"] > qc["last_scraped"]).sum()
)

print(
    "first_review after actual last_scraped:",
    (qc["first_review"] > qc["last_scraped"]).sum()
)

In [ ]:
# ============================================================
# STEP 11.4 — CHECK ACTUAL SCRAPE-DATE OFFSET
# ============================================================

scrape_qc = qc.copy()

scrape_qc["scrape_offset_days"] = (
    scrape_qc["last_scraped"]
    - scrape_qc["snapshot_date"]
).dt.days

print("=== ACTUAL SCRAPE DATE RANGE ===")

print(
    "Nominal snapshot date:",
    scrape_qc["snapshot_date"].iloc[0].date()
)

print(
    "Earliest last_scraped:",
    scrape_qc["last_scraped"].min()
)

print(
    "Latest last_scraped:",
    scrape_qc["last_scraped"].max()
)

print("\n=== SCRAPE OFFSET DAYS ===")

display(
    scrape_qc["scrape_offset_days"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("n_listings")
)

print("\n=== SUMMARY ===")

print(
    scrape_qc["scrape_offset_days"]
    .describe()
)

In [ ]:
# ============================================================
# STEP 11.5 — AUDIT ACTUAL SCRAPE DATES ACROSS 11 SNAPSHOTS
# ============================================================

scrape_audit = []

for snapshot_date in SNAPSHOT_DATES:

    file_path = (
        RAW_AIRBNB /
        snapshot_date /
        "listings.csv.gz"
    )

    temp = pd.read_csv(
        file_path,
        usecols=[
            "id",
            "last_scraped",
            "first_review",
            "last_review"
        ],
        low_memory=False
    )

    nominal_date = pd.Timestamp(snapshot_date)

    temp["last_scraped"] = pd.to_datetime(
        temp["last_scraped"],
        errors="coerce"
    )

    temp["first_review"] = pd.to_datetime(
        temp["first_review"],
        errors="coerce"
    )

    temp["last_review"] = pd.to_datetime(
        temp["last_review"],
        errors="coerce"
    )

    offset = (
        temp["last_scraped"] - nominal_date
    ).dt.days

    scrape_audit.append(
        {
            "snapshot_date": snapshot_date,
            "n_listings": len(temp),

            "earliest_last_scraped":
                temp["last_scraped"].min(),

            "latest_last_scraped":
                temp["last_scraped"].max(),

            "median_offset_days":
                offset.median(),

            "min_offset_days":
                offset.min(),

            "max_offset_days":
                offset.max(),

            "last_review_after_nominal_n":
                (
                    temp["last_review"]
                    > nominal_date
                ).sum(),

            "last_review_after_scrape_n":
                (
                    temp["last_review"]
                    > temp["last_scraped"]
                ).sum(),

            "first_review_after_scrape_n":
                (
                    temp["first_review"]
                    > temp["last_scraped"]
                ).sum()
        }
    )

scrape_audit = pd.DataFrame(scrape_audit)

display(scrape_audit)

print("\n=== CRITICAL QC ===")

print(
    "Snapshots with last_review after actual scrape:",
    (
        scrape_audit[
            "last_review_after_scrape_n"
        ] > 0
    ).sum()
)

print(
    "Snapshots with first_review after actual scrape:",
    (
        scrape_audit[
            "first_review_after_scrape_n"
        ] > 0
    ).sum()
)

In [ ]:
%run 02_clean_listings.py

In [ ]:
# ============================================================
# STEP 11.7 — INSPECT CLEANED OUTPUT SCHEMA
# ============================================================

test_clean = pd.read_parquet(
    INT_LISTINGS / "listings_2025-10-05.parquet"
)

print("=== CLEANED LISTINGS OUTPUT SCHEMA ===")
print("Rows:", len(test_clean))
print("Columns:", len(test_clean.columns))

for i, col in enumerate(test_clean.columns, start=1):
    print(f"{i:02d}. {col}")

In [ ]:
# ============================================================
# STEP 11.8 — VERIFY CLEANED SCHEMA AGAINST FROZEN DESIGN
# ============================================================

schema = pd.read_csv(
    INTERMEDIATE / "airbnb_listings_85_variable_screening.csv"
)

keep_vars = set(
    schema.loc[
        schema["decision"] == "KEEP",
        "column_name"
    ]
)

derive_source_vars = set(
    schema.loc[
        schema["decision"] == "DERIVE",
        "column_name"
    ]
)

processing_vars = set(
    schema.loc[
        schema["decision"] == "PROCESSING_ONLY",
        "column_name"
    ]
)

clean_cols = set(test_clean.columns)

# id was renamed to listing_id
processing_expected = {
    "listing_id" if x == "id" else x
    for x in processing_vars
}

derived_output_vars = {
    "amenities_count",
    "listing_review_age_days",
    "days_since_last_review"
}

technical_vars = {
    "snapshot_date",
    "price_raw"
}

expected_clean_cols = (
    keep_vars
    | derive_source_vars
    | processing_expected
    | derived_output_vars
    | technical_vars
)

print("=== STEP 11.8 — CLEANED SCHEMA QC ===")

print("KEEP source variables:", len(keep_vars))
print("DERIVE source variables:", len(derive_source_vars))
print("PROCESSING_ONLY variables:", len(processing_expected))

print("\nExpected cleaned columns:", len(expected_clean_cols))
print("Actual cleaned columns:", len(clean_cols))

unexpected = sorted(clean_cols - expected_clean_cols)
missing = sorted(expected_clean_cols - clean_cols)

print("\nUnexpected columns:")
print(unexpected)

print("\nMissing expected columns:")
print(missing)

print("\nQC:")
print("Unexpected n:", len(unexpected))
print("Missing n:", len(missing))

if len(unexpected) == 0 and len(missing) == 0:
    print("\nSCHEMA STATUS: PASS")
else:
    print("\nSCHEMA STATUS: REVIEW REQUIRED")

In [ ]:
%run 02_clean_listings.py

In [ ]:
# ============================================================
# STEP 11.10 — FINAL LISTINGS PARQUET INTEGRITY CHECK
# ============================================================

parquet_qc = []

for snapshot_date in SNAPSHOT_DATES:

    file_path = (
        INT_LISTINGS /
        f"listings_{snapshot_date}.parquet"
    )

    exists = file_path.exists()

    if exists:

        temp = pd.read_parquet(file_path)

        parquet_qc.append({
            "snapshot_date": snapshot_date,
            "file_exists": True,
            "rows": len(temp),
            "columns": len(temp.columns),
            "unique_listing_ids":
                temp["listing_id"].nunique(),
            "duplicate_listing_ids":
                temp["listing_id"].duplicated().sum()
        })

    else:

        parquet_qc.append({
            "snapshot_date": snapshot_date,
            "file_exists": False,
            "rows": None,
            "columns": None,
            "unique_listing_ids": None,
            "duplicate_listing_ids": None
        })


parquet_qc = pd.DataFrame(parquet_qc)

display(parquet_qc)


print("\n=== FINAL PARQUET QC ===")

print(
    "Files found:",
    parquet_qc["file_exists"].sum(),
    "/ 11"
)

print(
    "Total rows:",
    parquet_qc["rows"].sum()
)

print(
    "Files with 40 columns:",
    (parquet_qc["columns"] == 40).sum(),
    "/ 11"
)

print(
    "Files with duplicate IDs:",
    (parquet_qc["duplicate_listing_ids"] > 0).sum()
)

In [ ]:
# ============================================================
# STEP 12.5 — TERRITORIAL AUTHORITY 2025 RAW QC
# ============================================================

import geopandas as gpd
from pathlib import Path

from config import PROJECT_ROOT

TA_DIR = (
    PROJECT_ROOT
    / "01_raw"
    / "geography"
    / "statsnz-territorial-authority-2025-SHP"
)

TA_FILE = TA_DIR / "territorial-authority-2025.shp"

print("TA file path:")
print(TA_FILE)

print("\nFile exists:", TA_FILE.exists())

ta = gpd.read_file(TA_FILE)

print("\n=== TERRITORIAL AUTHORITY 2025 QC ===")
print("Number of rows:", len(ta))
print("CRS:", ta.crs)

print("\nGeometry types:")
print(ta.geometry.geom_type.value_counts())

print("\nColumns:")
print(ta.columns.tolist())

print("\nFirst 5 records:")
display(ta.head())

In [ ]:
print("=== TA 2025 ATTRIBUTE CHECK ===")

for col in ["TA2025_V1_", "TA2025_V_1", "TA2025_V_2"]:
    print(f"\nCOLUMN: {col}")
    print("Unique values:", ta[col].nunique())
    print(ta[col].tolist())

In [ ]:
# ============================================================
# STEP 12.7 — TA 2025 GEOMETRY INTEGRITY QC
# ============================================================

print("=== TA 2025 GEOMETRY INTEGRITY QC ===")

print("Total records:", len(ta))
print("Missing geometry:", ta.geometry.isna().sum())
print("Empty geometry:", ta.geometry.is_empty.sum())
print("Valid geometry:", ta.geometry.is_valid.sum())
print("Invalid geometry:", (~ta.geometry.is_valid).sum())

print("\nTA 999 check:")
display(
    ta.loc[
        ta["TA2025_V1_"] == "999",
        ["TA2025_V1_", "TA2025_V_1", "TA2025_V_2"]
    ]
)

In [ ]:
# ============================================================
# STEP 12.8 — AIRBNB COORDINATE QC BEFORE SPATIAL JOIN
# ============================================================

coord_qc = []

for snapshot_date in SNAPSHOT_DATES:

    file_path = (
        INT_LISTINGS /
        f"listings_{snapshot_date}.parquet"
    )

    df = pd.read_parquet(
        file_path,
        columns=["listing_id", "latitude", "longitude"]
    )

    coord_qc.append({
        "snapshot_date": snapshot_date,
        "rows": len(df),

        "missing_latitude":
            df["latitude"].isna().sum(),

        "missing_longitude":
            df["longitude"].isna().sum(),

        "latitude_min":
            df["latitude"].min(),

        "latitude_max":
            df["latitude"].max(),

        "longitude_min":
            df["longitude"].min(),

        "longitude_max":
            df["longitude"].max(),

        # Broad NZ sanity bounds only — not used for TA assignment
        "outside_nz_sanity_bounds": (
            (df["latitude"] < -48) |
            (df["latitude"] > -33) |
            (df["longitude"] < 165) |
            (df["longitude"] > 180)
        ).sum()
    })

coord_qc = pd.DataFrame(coord_qc)

display(coord_qc)

print("\n=== COORDINATE QC SUMMARY ===")
print("Snapshots checked:", len(coord_qc))
print("Total missing latitude:", coord_qc["missing_latitude"].sum())
print("Total missing longitude:", coord_qc["missing_longitude"].sum())
print(
    "Total outside broad NZ sanity bounds:",
    coord_qc["outside_nz_sanity_bounds"].sum()
)

In [ ]:
# ============================================================
# STEP 12.9 — INSPECT FLAGGED COORDINATES
# ============================================================

flagged_records = []

for snapshot_date in SNAPSHOT_DATES:

    file_path = (
        INT_LISTINGS /
        f"listings_{snapshot_date}.parquet"
    )

    df = pd.read_parquet(
        file_path,
        columns=["listing_id", "latitude", "longitude"]
    )

    flagged = df[
        (df["latitude"] < -48) |
        (df["latitude"] > -33) |
        (df["longitude"] < 165) |
        (df["longitude"] > 180)
    ].copy()

    flagged["snapshot_date"] = snapshot_date
    flagged_records.append(flagged)

flagged_records = pd.concat(
    flagged_records,
    ignore_index=True
)

print("=== FLAGGED COORDINATE INSPECTION ===")
print("Total flagged observations:", len(flagged_records))
print("Unique listing IDs:", flagged_records["listing_id"].nunique())

print("\nCoordinate ranges:")
print(
    flagged_records[
        ["latitude", "longitude"]
    ].agg(["min", "max"])
)

print("\nUnique coordinates:")
display(
    flagged_records[
        ["latitude", "longitude"]
    ]
    .drop_duplicates()
    .sort_values(["latitude", "longitude"])
)

In [ ]:
# ============================================================
# STEP 13.1 — TEST TA 2025 SPATIAL JOIN ON FIRST SNAPSHOT
# ============================================================

import geopandas as gpd
import pandas as pd

test_snapshot = "2025-10-05"

# ------------------------------------------------------------
# 1. Load cleaned Airbnb coordinates
# ------------------------------------------------------------

listing_file = (
    INT_LISTINGS /
    f"listings_{test_snapshot}.parquet"
)

df_test = pd.read_parquet(
    listing_file,
    columns=["listing_id", "latitude", "longitude"]
)

print("Input Airbnb rows:", len(df_test))


# ------------------------------------------------------------
# 2. Convert Airbnb coordinates to spatial points
#    Airbnb coordinates are WGS84
# ------------------------------------------------------------

gdf_test = gpd.GeoDataFrame(
    df_test,
    geometry=gpd.points_from_xy(
        df_test["longitude"],
        df_test["latitude"]
    ),
    crs="EPSG:4326"
)

print("Airbnb original CRS:", gdf_test.crs)


# ------------------------------------------------------------
# 3. Reproject to official TA 2025 CRS
# ------------------------------------------------------------

gdf_test = gdf_test.to_crs(ta.crs)

print("Airbnb projected CRS:", gdf_test.crs)
print("TA CRS:", ta.crs)


# ------------------------------------------------------------
# 4. Keep required TA fields only
# ------------------------------------------------------------

ta_join = ta[
    [
        "TA2025_V1_",
        "TA2025_V_1",
        "TA2025_V_2",
        "geometry"
    ]
].copy()


# ------------------------------------------------------------
# 5. Point-in-polygon spatial join
# ------------------------------------------------------------

joined_test = gpd.sjoin(
    gdf_test,
    ta_join,
    how="left",
    predicate="within"
)


# ------------------------------------------------------------
# 6. Basic QC
# ------------------------------------------------------------

print("\n=== TEST SPATIAL JOIN QC ===")

print("Input rows:", len(df_test))
print("Output rows:", len(joined_test))

print(
    "Matched to TA:",
    joined_test["TA2025_V1_"].notna().sum()
)

print(
    "Unmatched:",
    joined_test["TA2025_V1_"].isna().sum()
)

print(
    "Unique TA codes assigned:",
    joined_test["TA2025_V1_"].nunique()
)

print("\nUnmatched coordinates:")
display(
    joined_test.loc[
        joined_test["TA2025_V1_"].isna(),
        ["listing_id", "latitude", "longitude"]
    ]
)

In [ ]:
# ============================================================
# STEP 13.2 — AUDIT TA 999 AND CHATHAM ISLANDS
# ============================================================

print("=== TA 999 AUDIT ===")

ta999 = joined_test[
    joined_test["TA2025_V1_"].astype(str).str.strip() == "999"
].copy()

print("Listings assigned to TA 999:", len(ta999))

display(
    ta999[
        [
            "listing_id",
            "latitude",
            "longitude",
            "TA2025_V1_",
            "TA2025_V_1"
        ]
    ]
)


print("\n=== CHATHAM ISLANDS AUDIT ===")

chatham = joined_test[
    joined_test["TA2025_V_1"]
    .astype(str)
    .str.contains("Chatham", case=False, na=False)
].copy()

print("Listings assigned to Chatham Islands:", len(chatham))

display(
    chatham[
        [
            "listing_id",
            "latitude",
            "longitude",
            "TA2025_V1_",
            "TA2025_V_1"
        ]
    ]
)

In [ ]:
print(nearest_ta999.columns.tolist())

In [ ]:
# ============================================================
# STEP 13.3B — REVIEW TA 999 NEAREST-TA RESULTS
# ============================================================

print("=== TA 999 NEAREST-ACTUAL-TA AUDIT ===")
print("TA 999 listings:", len(ta999_points))
print("Nearest-join rows:", len(nearest_ta999))

print("\nDistance to nearest actual TA polygon (metres):")

print(
    nearest_ta999["distance_to_actual_ta_m"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nDistance groups:")

d = nearest_ta999["distance_to_actual_ta_m"]

print("<= 100 m:", (d <= 100).sum())

print(
    "101–500 m:",
    ((d > 100) & (d <= 500)).sum()
)

print(
    "501–1000 m:",
    ((d > 500) & (d <= 1000)).sum()
)

print(
    "> 1000 m:",
    (d > 1000).sum()
)

print("\nNearest actual TA assignments:")

display(
    nearest_ta999[
        [
            "listing_id",
            "latitude",
            "longitude",
            "TA2025_V1_",
            "TA2025_V_1",
            "distance_to_actual_ta_m"
        ]
    ]
    .sort_values("distance_to_actual_ta_m")
)

In [ ]:
# ============================================================
# STEP 13.4 — INSPECT TA 999 LISTINGS > 100 M FROM ACTUAL TA
# ============================================================

far_ta999 = nearest_ta999[
    nearest_ta999["distance_to_actual_ta_m"] > 100
].copy()

print("=== TA 999 LISTINGS > 100 M FROM ACTUAL TA ===")
print("Number of listings:", len(far_ta999))

display(
    far_ta999[
        [
            "listing_id",
            "latitude",
            "longitude",
            "TA2025_V1_",
            "TA2025_V_1",
            "distance_to_actual_ta_m"
        ]
    ]
    .sort_values(
        "distance_to_actual_ta_m",
        ascending=False
    )
)

In [ ]:
# ============================================================
# STEP 13.5 — FINAL QC OF TA 999 NEAREST REASSIGNMENT
# ============================================================

print("=== TA 999 REASSIGNMENT QC ===")

n_total = len(joined_test)
n_999 = len(nearest_ta999)

print("Total listings:", n_total)
print("TA 999 listings:", n_999)
print(
    "TA 999 share (%):",
    round(n_999 / n_total * 100, 3)
)

print("\nNearest actual TA distribution:")

ta999_distribution = (
    nearest_ta999
    .groupby(
        ["TA2025_V1_", "TA2025_V_1"],
        dropna=False
    )
    .size()
    .reset_index(name="n_listings")
    .sort_values("n_listings", ascending=False)
)

display(ta999_distribution)

print("\nDistance QC:")
print(
    nearest_ta999["distance_to_actual_ta_m"]
    .describe()
)

In [ ]:
# ============================================================
# STEP 13.6 — TA 2025 ASSIGNMENT QC ACROSS ALL 11 SNAPSHOTS
# Official point-in-polygon only
# TA 999 is retained as an official category
# ============================================================

ta_qc_all = []

# TA fields required for spatial join
ta_join = ta[
    [
        "TA2025_V1_",
        "TA2025_V_1",
        "TA2025_V_2",
        "geometry"
    ]
].copy()


for snapshot_date in SNAPSHOT_DATES:

    file_path = (
        INT_LISTINGS /
        f"listings_{snapshot_date}.parquet"
    )

    # --------------------------------------------------------
    # Load only fields needed for geography QC
    # --------------------------------------------------------
    df = pd.read_parquet(
        file_path,
        columns=["listing_id", "latitude", "longitude"]
    )

    # --------------------------------------------------------
    # Create WGS84 points
    # --------------------------------------------------------
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(
            df["longitude"],
            df["latitude"]
        ),
        crs="EPSG:4326"
    )

    # --------------------------------------------------------
    # Reproject to TA 2025 CRS
    # --------------------------------------------------------
    gdf = gdf.to_crs(ta.crs)

    # --------------------------------------------------------
    # Official point-in-polygon assignment
    # --------------------------------------------------------
    joined = gpd.sjoin(
        gdf,
        ta_join,
        how="left",
        predicate="within"
    )

    # Standardise TA code for QC
    ta_code = (
        joined["TA2025_V1_"]
        .astype("string")
        .str.strip()
    )

    # --------------------------------------------------------
    # QC statistics
    # --------------------------------------------------------
    n_rows = len(df)
    n_output = len(joined)
    n_unique_ids = joined["listing_id"].nunique()

    n_unmatched = joined["TA2025_V1_"].isna().sum()
    n_999 = (ta_code == "999").sum()
    n_chatham = (ta_code == "067").sum()

    n_actual_ta = (
        ta_code[
            ta_code.notna() &
            (ta_code != "999")
        ]
        .nunique()
    )

    ta_qc_all.append({
        "snapshot_date": snapshot_date,
        "input_rows": n_rows,
        "output_rows": n_output,
        "unique_listing_ids": n_unique_ids,
        "unmatched": n_unmatched,
        "ta_999": n_999,
        "ta_999_pct": round(n_999 / n_rows * 100, 3),
        "chatham_067": n_chatham,
        "actual_ta_codes_represented": n_actual_ta
    })


# ============================================================
# SUMMARY
# ============================================================

ta_qc_all = pd.DataFrame(ta_qc_all)

display(ta_qc_all)

print("\n=== ALL-SNAPSHOT TA ASSIGNMENT QC ===")

print("Snapshots checked:", len(ta_qc_all))

print(
    "Total input rows:",
    ta_qc_all["input_rows"].sum()
)

print(
    "Total output rows:",
    ta_qc_all["output_rows"].sum()
)

print(
    "Total unmatched:",
    ta_qc_all["unmatched"].sum()
)

print(
    "Total TA 999 observations:",
    ta_qc_all["ta_999"].sum()
)

print(
    "Overall TA 999 share (%):",
    round(
        ta_qc_all["ta_999"].sum()
        / ta_qc_all["input_rows"].sum()
        * 100,
        3
    )
)

print(
    "Total Chatham Islands observations:",
    ta_qc_all["chatham_067"].sum()
)

print(
    "Snapshots with row-count mismatch:",
    (
        ta_qc_all["input_rows"]
        != ta_qc_all["output_rows"]
    ).sum()
)

print(
    "Snapshots with duplicate listing IDs after join:",
    (
        ta_qc_all["output_rows"]
        != ta_qc_all["unique_listing_ids"]
    ).sum()
)

In [ ]:
# ============================================================
# STEP 13.7 — PRODUCTION TA 2025 GEOGRAPHY ASSIGNMENT
# ============================================================

from pathlib import Path

GEO_OUT = PROJECT_ROOT / "02_intermediate" / "geography"
GEO_OUT.mkdir(parents=True, exist_ok=True)

production_qc = []

ta_join = ta[
    [
        "TA2025_V1_",
        "TA2025_V_1",
        "TA2025_V_2",
        "geometry"
    ]
].copy()


for snapshot_date in SNAPSHOT_DATES:

    input_file = (
        INT_LISTINGS /
        f"listings_{snapshot_date}.parquet"
    )

    df = pd.read_parquet(
        input_file,
        columns=["listing_id", "latitude", "longitude"]
    )

    # Airbnb coordinates: WGS84
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(
            df["longitude"],
            df["latitude"]
        ),
        crs="EPSG:4326"
    )

    # Reproject to official TA 2025 CRS
    gdf = gdf.to_crs(ta.crs)

    # Official point-in-polygon assignment
    joined = gpd.sjoin(
        gdf,
        ta_join,
        how="left",
        predicate="within"
    )

    # Standardised output
    out = joined[
        [
            "listing_id",
            "TA2025_V1_",
            "TA2025_V_1",
            "TA2025_V_2"
        ]
    ].copy()

    out = out.rename(
        columns={
            "TA2025_V1_": "ta_code",
            "TA2025_V_1": "ta_name",
            "TA2025_V_2": "ta_name_ascii"
        }
    )

    out["snapshot_date"] = snapshot_date

    # Keep official code as string
    out["ta_code"] = (
        out["ta_code"]
        .astype("string")
        .str.strip()
    )

    # Reorder
    out = out[
        [
            "snapshot_date",
            "listing_id",
            "ta_code",
            "ta_name",
            "ta_name_ascii"
        ]
    ]

    # Save
    output_file = (
        GEO_OUT /
        f"listing_ta_{snapshot_date}.parquet"
    )

    out.to_parquet(
        output_file,
        index=False
    )

    production_qc.append({
        "snapshot_date": snapshot_date,
        "input_rows": len(df),
        "output_rows": len(out),
        "unique_listing_ids": out["listing_id"].nunique(),
        "missing_ta": out["ta_code"].isna().sum(),
        "ta_999": (out["ta_code"] == "999").sum(),
        "chatham_067": (out["ta_code"] == "067").sum()
    })

    print(
        f"{snapshot_date}: saved {len(out):,} rows"
    )


# ============================================================
# PRODUCTION QC
# ============================================================

production_qc = pd.DataFrame(production_qc)

display(production_qc)

print("\n=== PRODUCTION GEOGRAPHY QC ===")

print("Files created:", len(production_qc))
print("Total rows:", production_qc["output_rows"].sum())
print("Total missing TA:", production_qc["missing_ta"].sum())
print("Total TA 999:", production_qc["ta_999"].sum())
print("Total Chatham 067:", production_qc["chatham_067"].sum())

print(
    "Snapshots with row-count mismatch:",
    (
        production_qc["input_rows"]
        != production_qc["output_rows"]
    ).sum()
)

print(
    "Snapshots with duplicate IDs:",
    (
        production_qc["output_rows"]
        != production_qc["unique_listing_ids"]
    ).sum()
)

In [ ]:
# ============================================================
# STEP 13.8 — TA 999 UNIQUE LISTING QC
# ============================================================

ta999_all = []

for snapshot_date in SNAPSHOT_DATES:

    geo_file = (
        GEO_OUT /
        f"listing_ta_{snapshot_date}.parquet"
    )

    df_geo = pd.read_parquet(
        geo_file,
        columns=[
            "snapshot_date",
            "listing_id",
            "ta_code"
        ]
    )

    # Keep official Area Outside Territorial Authority only
    df_999 = df_geo[
        df_geo["ta_code"].astype("string").str.strip() == "999"
    ].copy()

    ta999_all.append(df_999)


# Combine all 11 snapshots
ta999_all = pd.concat(
    ta999_all,
    ignore_index=True
)


# ============================================================
# BASIC QC
# ============================================================

print("=== TA 999 UNIQUE LISTING QC ===")

print(
    "Total TA 999 listing-snapshot observations:",
    len(ta999_all)
)

print(
    "Unique Airbnb listing IDs:",
    ta999_all["listing_id"].nunique()
)


# ============================================================
# NUMBER OF SNAPSHOTS PER UNIQUE LISTING
# ============================================================

listing_frequency = (
    ta999_all
    .groupby("listing_id")
    .agg(
        n_snapshots=("snapshot_date", "nunique"),
        first_snapshot=("snapshot_date", "min"),
        last_snapshot=("snapshot_date", "max")
    )
    .reset_index()
)

print("\nSnapshots represented per TA 999 listing:")

display(
    listing_frequency["n_snapshots"]
    .value_counts()
    .sort_index()
    .rename_axis("n_snapshots")
    .reset_index(name="n_unique_listings")
)


print("\nSummary of snapshot frequency:")

print(
    listing_frequency["n_snapshots"].describe()
)


# ============================================================
# LISTINGS APPEARING MOST FREQUENTLY AS TA 999
# ============================================================

print("\nListings with highest number of TA 999 appearances:")

display(
    listing_frequency
    .sort_values(
        ["n_snapshots", "listing_id"],
        ascending=[False, True]
    )
    .head(30)
)